# 03 - Automatic Colour Segmentation for the Full Traffic-Sign Dataset

This notebook combines the existing **red**, **blue**, **yellow**, and **shape-detection** processes into one automatic dataset pipeline. Each image is evaluated by all three colour processes. The strongest colour-and-shape candidate selects the correct segmentation process.

Outputs:
- Segmented copies are saved outside the original dataset.
- Original filenames and relative subfolders are preserved.
- `segmentation_results.csv` records the selected colour, shape, score, bounding box, and status.
- `annotations_segmented.csv` keeps the synchronized annotations and adds segmentation results.
- Seven visualization stages show up to 30 examples: 10 red, 10 blue, and 10 yellow.

> The source dataset and its original `annotations.csv` are never modified.

## 1. Install and import libraries

In [ ]:
%pip install -q opencv-python pandas numpy matplotlib tqdm

In [ ]:
from pathlib import Path
import random
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
print("OpenCV version:", cv2.__version__)

## 2. Configuration

The paths match notebooks `01` and `02`. Change only this cell if your project folders are different.

In [ ]:
DATA_DIR = Path("../data")
DATASET_DIR = DATA_DIR / "chinese_traffic_signs"
OUTPUT_ROOT = DATA_DIR / "chinese_traffic_signs_segmented"
OUTPUT_IMAGES_DIR = OUTPUT_ROOT / "images"
RESULTS_CSV = OUTPUT_ROOT / "segmentation_results.csv"
OUTPUT_ANNOTATIONS_CSV = OUTPUT_ROOT / "annotations_segmented.csv"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".tif", ".tiff"}
SAMPLE_COUNT_PER_COLOUR = 10
RANDOM_SEED = 42

# None processes the full synchronized dataset. Use an integer only for a quick test.
PROCESS_LIMIT = None

# False saves a full-size image with the non-sign background blacked out.
# True saves only the detected bounding-box crop.
SAVE_CROPPED_IMAGE = False

# If detection fails, preserve the original image in the output dataset and log it.
# This keeps the output image count synchronized with annotations.csv.
SAVE_ORIGINAL_ON_FAILURE = True

OUTPUT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset folder       :", DATASET_DIR.resolve())
print("Output image folder :", OUTPUT_IMAGES_DIR.resolve())
print("Save cropped images :", SAVE_CROPPED_IMAGE)

## 3. Load the synchronized annotations and image paths

This verification follows notebook `02`: one annotation row must match one image filename.

In [ ]:
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset folder was not found: {DATASET_DIR.resolve()}\n"
        "Run notebooks 01 and 02 first, or correct DATASET_DIR."
    )

required_columns = {"file_name", "category"}
annotation_candidates = [DATASET_DIR / "annotations.csv"]
annotation_candidates.extend(sorted(DATASET_DIR.rglob("*.csv")))

ANNOTATIONS_FILE = None
annotations_df = None

for candidate in dict.fromkeys(annotation_candidates):
    if not candidate.exists():
        continue
    try:
        candidate_df = pd.read_csv(candidate)
    except Exception:
        continue
    if required_columns.issubset(candidate_df.columns):
        ANNOTATIONS_FILE = candidate
        annotations_df = candidate_df.copy()
        break

if ANNOTATIONS_FILE is None:
    raise FileNotFoundError(
        "No compatible annotation CSV was found. "
        f"Expected columns: {sorted(required_columns)}"
    )

annotations_df["file_name"] = annotations_df["file_name"].astype(str).str.strip()

image_paths = sorted(
    path for path in DATASET_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

filename_lookup = {}
duplicate_image_names = []
for image_path in image_paths:
    key = image_path.name.casefold()
    if key in filename_lookup:
        duplicate_image_names.append(image_path.name)
    else:
        filename_lookup[key] = image_path

if duplicate_image_names:
    raise RuntimeError(
        "Duplicate image filenames remain. Run notebook 02 again. Examples: "
        + str(duplicate_image_names[:10])
    )

if annotations_df["file_name"].str.casefold().duplicated().any():
    raise RuntimeError("Duplicate annotation filenames remain. Run notebook 02 again.")

annotations_df["source_path"] = annotations_df["file_name"].str.casefold().map(filename_lookup)
missing_rows = annotations_df[annotations_df["source_path"].isna()]

if not missing_rows.empty:
    raise RuntimeError(
        f"{len(missing_rows)} annotation rows have no matching image. "
        "Run notebook 02 again before segmentation."
    )

if len(image_paths) != len(annotations_df):
    raise RuntimeError(
        f"Image count ({len(image_paths)}) and annotation count "
        f"({len(annotations_df)}) are different. Run notebook 02 again."
    )

processing_df = annotations_df.copy()
if PROCESS_LIMIT is not None:
    processing_df = processing_df.head(int(PROCESS_LIMIT)).copy()

print("Annotation file :", ANNOTATIONS_FILE.resolve())
print("Images verified :", len(image_paths))
print("Rows to process :", len(processing_df))
display(processing_df[["file_name", "category"]].head())

## 4. Shared mask utilities

In [ ]:
def remove_small_components(binary_mask, min_area_ratio):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    cleaned_mask = np.zeros_like(binary_mask)
    min_area = binary_mask.shape[0] * binary_mask.shape[1] * min_area_ratio

    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned_mask[labels == label] = 255

    return cleaned_mask


def valid_adaptive_block_size(channel, preferred_size):
    block_size = min(int(preferred_size), min(channel.shape[:2]))
    if block_size % 2 == 0:
        block_size -= 1
    return max(block_size, 3)


def empty_colour_result(image_bgr):
    empty_mask = np.zeros(image_bgr.shape[:2], dtype=np.uint8)
    return {
        "preprocessed": image_bgr.copy(),
        "colour_mask": empty_mask.copy(),
        "cleaned_mask": empty_mask.copy(),
    }

## 5. Red segmentation process

Preserves the red notebook's Gaussian blur, HSV redness map, Otsu thresholding, opening, and closing.

In [ ]:
def segment_red_stages(image_bgr):
    blurred = cv2.GaussianBlur(image_bgr, (3, 3), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

    hue = hsv[:, :, 0].astype(np.float32)
    saturation = hsv[:, :, 1].astype(np.float32)

    distance_from_red = np.minimum(hue, 180.0 - hue)
    redness = np.maximum(0.0, 15.0 - distance_from_red) / 15.0
    saturation_factor = np.clip((saturation - 30.0) / 50.0, 0.0, 1.0)

    difference_map = redness * saturation_factor * 255.0
    if np.max(difference_map) > np.min(difference_map):
        difference_map = cv2.normalize(difference_map, None, 0, 255, cv2.NORM_MINMAX)
    difference_map = difference_map.astype(np.uint8)

    _, raw_mask = cv2.threshold(
        difference_map, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opened = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, open_kernel)
    cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, close_kernel)

    return {
        "preprocessed": blurred,
        "colour_response": difference_map,
        "colour_mask": raw_mask,
        "cleaned_mask": cleaned,
    }

## 6. Blue segmentation process

Preserves the blue notebook's bilateral filtering, HSV masks, adaptive saturation threshold, component removal, and morphological closing.

In [ ]:
def segment_blue_stages(image_bgr):
    filtered = cv2.bilateralFilter(image_bgr, d=7, sigmaColor=40, sigmaSpace=40)
    hsv = cv2.cvtColor(filtered, cv2.COLOR_BGR2HSV)
    hue, saturation, value = cv2.split(hsv)

    hue_mask = cv2.inRange(hue, 99, 125)
    block_size = valid_adaptive_block_size(saturation, 91)
    saturation_mask = cv2.adaptiveThreshold(
        saturation, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, block_size, 2
    )
    value_mask = cv2.inRange(value, 42, 255)

    raw_mask = cv2.bitwise_and(hue_mask, saturation_mask)
    raw_mask = cv2.bitwise_and(raw_mask, value_mask)

    component_mask = remove_small_components(raw_mask, min_area_ratio=0.05)
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (4, 4))
    cleaned = cv2.morphologyEx(
        component_mask, cv2.MORPH_CLOSE, close_kernel, iterations=1
    )

    return {
        "preprocessed": filtered,
        "colour_response": hue_mask,
        "colour_mask": raw_mask,
        "cleaned_mask": cleaned,
    }

## 7. Yellow segmentation process

Preserves the yellow notebook's optional CLAHE enhancement, HSV threshold, adaptive value threshold, and morphological closing.

In [ ]:
def clean_dark_yellow_image(image_bgr, brightness_threshold=140):
    cleaned = image_bgr.copy()
    hsv = cv2.cvtColor(cleaned, cv2.COLOR_BGR2HSV)
    hue, saturation, value = cv2.split(hsv)
    height, width = value.shape
    centre = value[height // 4: 3 * height // 4, width // 4: 3 * width // 4]

    if centre.size and float(np.mean(centre)) < brightness_threshold:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_value = clahe.apply(value)
        cleaned = cv2.cvtColor(
            cv2.merge((hue, saturation, enhanced_value)),
            cv2.COLOR_HSV2BGR
        )

    return cleaned


def segment_yellow_stages(image_bgr):
    cleaned_image = clean_dark_yellow_image(image_bgr)
    hsv = cv2.cvtColor(cleaned_image, cv2.COLOR_BGR2HSV)
    _, _, value = cv2.split(hsv)

    hsv_mask = cv2.inRange(
        hsv,
        np.array((5, 80, 30), dtype=np.uint8),
        np.array((40, 255, 255), dtype=np.uint8)
    )

    block_size = valid_adaptive_block_size(value, 51)
    adaptive_mask = cv2.adaptiveThreshold(
        value, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, block_size, 7
    )
    raw_mask = cv2.bitwise_and(hsv_mask, adaptive_mask)

    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(
        raw_mask, cv2.MORPH_CLOSE, close_kernel, iterations=1
    )

    return {
        "preprocessed": cleaned_image,
        "colour_response": hsv_mask,
        "colour_mask": raw_mask,
        "cleaned_mask": cleaned,
    }

## 8. Shape detection and automatic colour routing

Every colour mask produces a candidate. The selection score considers relative contour area, distance from the image centre, solidity, circularity, recognized shape, colour coverage, and HSV hue agreement. Relative area is used instead of the fixed 1,000-pixel threshold so small dataset images are supported.

In [ ]:
COLOUR_FUNCTIONS = {
    "red": segment_red_stages,
    "blue": segment_blue_stages,
    "yellow": segment_yellow_stages,
}


def extract_shape_features(contour):
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    if perimeter <= 0:
        return None

    circularity = 4.0 * np.pi * area / (perimeter * perimeter)
    epsilon_ratio = 0.018 if area < 3000 else 0.02
    approximation = cv2.approxPolyDP(contour, epsilon_ratio * perimeter, True)
    x, y, width, height = cv2.boundingRect(contour)

    return {
        "area": float(area),
        "perimeter": float(perimeter),
        "circularity": float(circularity),
        "vertices": int(len(approximation)),
        "aspect_ratio": float(width / height) if height else 0.0,
        "bounding_box": (int(x), int(y), int(width), int(height)),
        "approximation": approximation,
    }


def classify_shape(features):
    if features is None:
        return "Unknown"

    vertices = features["vertices"]
    circularity = features["circularity"]
    aspect_ratio = features["aspect_ratio"]

    if vertices == 3:
        return "Triangle"
    if vertices == 4:
        return "Square" if 0.9 <= aspect_ratio <= 1.1 else "Rectangle"
    if circularity >= 0.82:
        return "Circle"
    if 7 <= vertices <= 9:
        return "Octagon"
    return "Unknown"


def reconstruct_outer_contour(contour, solidity_threshold=0.90):
    if contour is None:
        return None
    hull = cv2.convexHull(contour)
    contour_area = cv2.contourArea(contour)
    hull_area = cv2.contourArea(hull)
    if hull_area <= 0:
        return contour
    if contour_area / hull_area < solidity_threshold:
        hull_perimeter = cv2.arcLength(hull, True)
        return cv2.approxPolyDP(hull, 0.01 * hull_perimeter, True)
    return contour


def hue_agreement_score(image_bgr, contour, colour):
    contour_mask = np.zeros(image_bgr.shape[:2], dtype=np.uint8)
    cv2.drawContours(contour_mask, [contour], -1, 255, cv2.FILLED)

    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    hue = hsv[:, :, 0].astype(np.float32)
    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]
    valid = (contour_mask > 0) & (saturation >= 45) & (value >= 30)

    if not np.any(valid):
        return 0.0

    selected_hue = hue[valid]
    if colour == "red":
        distance = np.minimum(selected_hue, 180.0 - selected_hue)
        affinity = np.clip(1.0 - distance / 18.0, 0.0, 1.0)
    elif colour == "yellow":
        affinity = np.clip(1.0 - np.abs(selected_hue - 25.0) / 20.0, 0.0, 1.0)
    else:
        affinity = np.clip(1.0 - np.abs(selected_hue - 112.0) / 25.0, 0.0, 1.0)

    return float(np.mean(affinity))


def find_best_candidate(image_bgr, colour, stages):
    mask = stages["cleaned_mask"]
    height, width = mask.shape
    image_area = float(height * width)
    image_diagonal = float(np.hypot(width, height))

    contours, _ = cv2.findContours(
        mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    best = None
    for contour in contours:
        area = cv2.contourArea(contour)
        area_ratio = area / image_area
        if area < max(20.0, image_area * 0.002) or area_ratio > 0.85:
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter <= 0:
            continue

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        if hull_area <= 0:
            continue

        moments = cv2.moments(contour)
        if moments["m00"] == 0:
            continue

        centre_x = moments["m10"] / moments["m00"]
        centre_y = moments["m01"] / moments["m00"]
        centre_distance = np.hypot(centre_x - width / 2, centre_y - height / 2)

        solidity = float(area / hull_area)
        circularity = float(np.clip(4.0 * np.pi * area / (perimeter * perimeter), 0, 1))
        area_score = float(min(area_ratio / 0.25, 1.0))
        centre_score = float(max(0.0, 1.0 - centre_distance / (0.5 * image_diagonal + 1e-6)))

        shape_features = extract_shape_features(contour)
        shape = classify_shape(shape_features)
        shape_score = 1.0 if shape != "Unknown" else 0.25

        filled_for_coverage = np.zeros_like(mask)
        cv2.drawContours(filled_for_coverage, [contour], -1, 255, cv2.FILLED)
        filled_pixels = max(cv2.countNonZero(filled_for_coverage), 1)
        colour_pixels = cv2.countNonZero(cv2.bitwise_and(stages["colour_mask"], filled_for_coverage))
        colour_coverage = float(colour_pixels / filled_pixels)
        hue_score = hue_agreement_score(image_bgr, contour, colour)

        score = (
            0.25 * area_score
            + 0.17 * centre_score
            + 0.13 * solidity
            + 0.10 * circularity
            + 0.15 * shape_score
            + 0.08 * min(colour_coverage / 0.35, 1.0)
            + 0.12 * hue_score
        )

        candidate = {
            "colour": colour,
            "contour": contour,
            "features": shape_features,
            "shape": shape,
            "score": float(score),
            "area_ratio": float(area_ratio),
            "solidity": solidity,
            "colour_coverage": colour_coverage,
            "hue_score": hue_score,
        }
        if best is None or candidate["score"] > best["score"]:
            best = candidate

    return best

In [ ]:
def process_one_image(image_bgr):
    all_stages = {}
    candidates = []

    for colour, segmentation_function in COLOUR_FUNCTIONS.items():
        try:
            stages = segmentation_function(image_bgr)
        except cv2.error as error:
            warnings.warn(f"{colour} segmentation failed: {error}")
            stages = empty_colour_result(image_bgr)

        all_stages[colour] = stages
        candidate = find_best_candidate(image_bgr, colour, stages)
        if candidate is not None:
            candidates.append(candidate)

    if not candidates:
        empty_mask = np.zeros(image_bgr.shape[:2], dtype=np.uint8)
        return {
            "detected": False,
            "colour": "undetected",
            "shape": "Unknown",
            "score": 0.0,
            "bbox": None,
            "original": image_bgr,
            "preprocessed": image_bgr.copy(),
            "colour_mask": empty_mask.copy(),
            "cleaned_mask": empty_mask.copy(),
            "detected_overlay": image_bgr.copy(),
            "filled_mask": empty_mask.copy(),
            "segmented": np.zeros_like(image_bgr),
            "cropped": np.zeros((1, 1, 3), dtype=np.uint8),
            "candidate_scores": {},
        }

    winner = max(candidates, key=lambda item: item["score"])
    colour = winner["colour"]
    stages = all_stages[colour]
    outer_contour = reconstruct_outer_contour(winner["contour"])

    features = extract_shape_features(outer_contour)
    shape = classify_shape(features)
    x, y, width, height = features["bounding_box"]

    filled_mask = np.zeros(image_bgr.shape[:2], dtype=np.uint8)
    cv2.drawContours(filled_mask, [outer_contour], -1, 255, cv2.FILLED)
    segmented = cv2.bitwise_and(image_bgr, image_bgr, mask=filled_mask)
    cropped = segmented[y:y + height, x:x + width].copy()

    overlay = image_bgr.copy()
    cv2.drawContours(overlay, [outer_contour], -1, (0, 255, 0), 2)
    label_y = max(y - 8, 18)
    cv2.putText(
        overlay, f"{colour.title()} | {shape}", (x, label_y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2, cv2.LINE_AA
    )

    return {
        "detected": True,
        "colour": colour,
        "shape": shape,
        "score": winner["score"],
        "bbox": (x, y, width, height),
        "original": image_bgr,
        "preprocessed": stages["preprocessed"],
        "colour_mask": stages["colour_mask"],
        "cleaned_mask": stages["cleaned_mask"],
        "detected_overlay": overlay,
        "filled_mask": filled_mask,
        "segmented": segmented,
        "cropped": cropped,
        "candidate_scores": {
            candidate["colour"]: candidate["score"] for candidate in candidates
        },
    }

## 9. Process and save the complete dataset

The sampling code uses reservoir sampling. Therefore, it can keep a random set of 10 examples per predicted colour without retaining every full-resolution intermediate image in memory.

In [ ]:
random_generator = random.Random(RANDOM_SEED)
sample_records = {"red": [], "blue": [], "yellow": []}
seen_per_colour = {"red": 0, "blue": 0, "yellow": 0}
result_rows = []
write_failures = []

for row in tqdm(
    processing_df.itertuples(index=False),
    total=len(processing_df),
    desc="Segmenting dataset"
):
    source_path = Path(row.source_path)
    relative_path = source_path.relative_to(DATASET_DIR)
    destination_path = OUTPUT_IMAGES_DIR / relative_path
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.imread(str(source_path))
    if image_bgr is None:
        result_rows.append({
            "file_name": row.file_name,
            "category": row.category,
            "relative_output_path": str(relative_path),
            "status": "unreadable",
            "selected_colour": "undetected",
            "detected_shape": "Unknown",
            "selection_score": 0.0,
            "bbox_x": np.nan, "bbox_y": np.nan,
            "bbox_width": np.nan, "bbox_height": np.nan,
            "red_score": np.nan, "blue_score": np.nan, "yellow_score": np.nan,
        })
        continue

    result = process_one_image(image_bgr)

    if result["detected"]:
        output_image = result["cropped"] if SAVE_CROPPED_IMAGE else result["segmented"]
        status = "segmented"
        x, y, width, height = result["bbox"]
    else:
        output_image = image_bgr if SAVE_ORIGINAL_ON_FAILURE else result["segmented"]
        status = "fallback_original" if SAVE_ORIGINAL_ON_FAILURE else "empty_output"
        x = y = width = height = np.nan

    if not cv2.imwrite(str(destination_path), output_image):
        write_failures.append(str(destination_path))
        status = "write_failed"

    scores = result["candidate_scores"]
    result_rows.append({
        "file_name": row.file_name,
        "category": row.category,
        "relative_output_path": str(relative_path),
        "status": status,
        "selected_colour": result["colour"],
        "detected_shape": result["shape"],
        "selection_score": result["score"],
        "bbox_x": x, "bbox_y": y,
        "bbox_width": width, "bbox_height": height,
        "red_score": scores.get("red", np.nan),
        "blue_score": scores.get("blue", np.nan),
        "yellow_score": scores.get("yellow", np.nan),
    })

    colour = result["colour"]
    if result["detected"] and colour in sample_records:
        seen_per_colour[colour] += 1
        sample_item = {
            "filename": row.file_name,
            "category": row.category,
            **result,
        }
        if len(sample_records[colour]) < SAMPLE_COUNT_PER_COLOUR:
            sample_records[colour].append(sample_item)
        else:
            replacement_index = random_generator.randrange(seen_per_colour[colour])
            if replacement_index < SAMPLE_COUNT_PER_COLOUR:
                sample_records[colour][replacement_index] = sample_item

results_df = pd.DataFrame(result_rows)
results_df.to_csv(RESULTS_CSV, index=False)

if write_failures:
    warnings.warn(f"{len(write_failures)} output images could not be written.")

print("Processing completed.")
print("Results CSV:", RESULTS_CSV.resolve())
print("Output images:", OUTPUT_IMAGES_DIR.resolve())
display(results_df.head())

## 10. Save synchronized output annotations and verify outputs

In [ ]:
annotation_columns = [column for column in annotations_df.columns if column != "source_path"]
output_annotations_df = annotations_df[annotation_columns].merge(
    results_df.drop(columns=["category"], errors="ignore"),
    on="file_name",
    how="left",
    validate="one_to_one"
)
output_annotations_df.to_csv(OUTPUT_ANNOTATIONS_CSV, index=False)

saved_image_paths = [
    path for path in OUTPUT_IMAGES_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

summary_df = pd.DataFrame([{
    "Rows processed": len(results_df),
    "Output images currently present": len(saved_image_paths),
    "Successfully segmented": int((results_df["status"] == "segmented").sum()),
    "Fallback originals": int((results_df["status"] == "fallback_original").sum()),
    "Unreadable images": int((results_df["status"] == "unreadable").sum()),
    "Write failures": int((results_df["status"] == "write_failed").sum()),
}])

display(summary_df)
display(pd.crosstab(results_df["selected_colour"], results_df["detected_shape"], margins=True))

if PROCESS_LIMIT is None and len(output_annotations_df) != len(results_df):
    raise RuntimeError("Output annotations and segmentation results have different lengths.")

if (results_df["status"] == "write_failed").any():
    raise RuntimeError("At least one segmented image failed to save. Review the rows above.")

print("Output annotation file:", OUTPUT_ANNOTATIONS_CSV.resolve())
print("Verification completed.")

## 11. Display 30 images for every processing stage

Each figure uses the same structure:
- Row 1: up to 10 red signs
- Row 2: up to 10 blue signs
- Row 3: up to 10 yellow signs

If fewer than 10 signs are automatically assigned to a colour, the unused positions remain blank.

In [ ]:
STAGE_DEFINITIONS = [
    ("original", "Stage 1 - Original images", False),
    ("preprocessed", "Stage 2 - Preprocessed images", False),
    ("colour_mask", "Stage 3 - Created colour masks", True),
    ("cleaned_mask", "Stage 4 - Cleaned masks", True),
    ("detected_overlay", "Stage 5 - Detected contour and shape", False),
    ("filled_mask", "Stage 6 - Filled contour masks", True),
    ("segmented", "Stage 7 - Final segmented images", False),
]


def display_stage_grid(sample_records, stage_key, figure_title, grayscale=False):
    colours = ["red", "blue", "yellow"]
    columns = SAMPLE_COUNT_PER_COLOUR
    fig, axes = plt.subplots(3, columns, figsize=(2.7 * columns, 8.5), squeeze=False)

    for row_index, colour in enumerate(colours):
        records = sample_records[colour]
        for column_index in range(columns):
            axis = axes[row_index, column_index]
            axis.axis("off")

            if column_index >= len(records):
                if column_index == 0:
                    axis.set_title(f"No {colour} samples", fontsize=9)
                continue

            record = records[column_index]
            image = record[stage_key]

            if grayscale:
                axis.imshow(image, cmap="gray", vmin=0, vmax=255)
            else:
                axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

            short_name = Path(record["filename"]).name
            axis.set_title(
                f"{short_name}\n{record['shape']} | {record['score']:.3f}",
                fontsize=7
            )

        axes[row_index, 0].set_ylabel(colour.upper(), fontsize=12, fontweight="bold")

    fig.suptitle(figure_title, fontsize=18, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


for stage_key, figure_title, grayscale in STAGE_DEFINITIONS:
    display_stage_grid(sample_records, stage_key, figure_title, grayscale)

## 12. Review uncertain and failed cases

These tables are important for tuning. A low selection score or a small difference between the two best colour scores means the automatic colour route is uncertain.

In [ ]:
score_columns = ["red_score", "blue_score", "yellow_score"]
available_scores = results_df[score_columns].fillna(-1).to_numpy()
sorted_scores = np.sort(available_scores, axis=1)
results_df["colour_score_margin"] = sorted_scores[:, -1] - sorted_scores[:, -2]

uncertain_df = results_df[
    (results_df["status"] != "segmented")
    | (results_df["selection_score"] < 0.45)
    | (results_df["colour_score_margin"] < 0.04)
].sort_values(["status", "selection_score"])

print("Cases recommended for manual review:", len(uncertain_df))
display(uncertain_df.head(50))

uncertain_df.to_csv(OUTPUT_ROOT / "segmentation_cases_to_review.csv", index=False)

## Interpretation

- **Created colour mask:** pixels selected by the colour-specific thresholding process.
- **Cleaned mask:** noise removed and broken sign regions connected.
- **Detected contour and shape:** selected boundary plus the shape classification result.
- **Filled contour mask:** the full sign interior, including white and black symbols.
- **Final segmented image:** the original traffic sign retained while the external background is black.

Review `segmentation_cases_to_review.csv` before extracting HOG features. If one colour is frequently incorrect, tune only that colour process and rerun this notebook.